# Food Image Classification with EfficientNetB3
### Transfer Learning on the `bjoernjostein/food-classification` Dataset (61 Classes)

## Prerequisites

Before running this notebook, ensure you have Kaggle API credentials configured:
1. Go to https://www.kaggle.com/settings/api
2. Click **Create New Token** — this downloads `kaggle.json`
3. Place `kaggle.json` at `~/.kaggle/kaggle.json` (Linux/Mac) or `C:\\Users\\<you>\\.kaggle\\kaggle.json` (Windows)
4. On Linux/Mac: `chmod 600 ~/.kaggle/kaggle.json`

The notebook uses `kagglehub` which reads these credentials automatically.

In [ ]:
!pip install kagglehub timm torch torchvision numpy pandas matplotlib seaborn scikit-learn Pillow tqdm --quiet

## Section 1 — Setup & Imports

In [ ]:
import os
import random
import warnings
import pathlib

import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, cohen_kappa_score
)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
import torchvision.transforms as T
import timm

warnings.filterwarnings('ignore')

In [ ]:
SEED         = 42
DATASET_SLUG = "bjoernjostein/food-classification"
IMG_SIZE     = (224, 224)
BATCH_SIZE   = 32
EPOCHS       = 5 if not torch.cuda.is_available() else 20
LR           = 1e-4
FINE_TUNE_LR = 1e-5
NUM_WORKERS  = 0   # 0 is safest on Windows; increase if on Linux
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Config: EPOCHS={EPOCHS}, BATCH_SIZE={BATCH_SIZE}, DEVICE={DEVICE}")
if not torch.cuda.is_available():
    print("⚠️  No GPU detected. Training will be slow. Consider using Google Colab or reducing EPOCHS.")
else:
    print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")

In [ ]:
def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
os.makedirs("figures", exist_ok=True)
print("Seed set. figures/ directory ready.")

## Section 2 — Data Download with kagglehub

In [ ]:
raw_path = kagglehub.dataset_download(DATASET_SLUG)
DATA_ROOT = pathlib.Path(raw_path)
print(f"Dataset downloaded to: {DATA_ROOT}")

In [ ]:
for root, dirs, files in os.walk(DATA_ROOT):
    level = str(root).replace(str(DATA_ROOT), '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{pathlib.Path(root).name}/')
    if level < 2:
        for f in files[:5]:
            print(f'{indent}  {f}')

In [ ]:
_IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

def find_split_dir(root: pathlib.Path, candidates: list) -> pathlib.Path | None:
    for name in candidates:
        p = root / name
        if p.is_dir():
            return p
    return None

TRAIN_DIR = find_split_dir(DATA_ROOT, ["train", "Train", "training"])
VAL_DIR   = find_split_dir(DATA_ROOT, ["valid", "validation", "val", "Valid"])
TEST_DIR  = find_split_dir(DATA_ROOT, ["test", "Test"])

assert TRAIN_DIR is not None, "Could not find a train/ directory in the dataset"
print(f"Train dir : {TRAIN_DIR}")
print(f"Val dir   : {VAL_DIR}")
print(f"Test dir  : {TEST_DIR}")

CLASS_NAMES = sorted([d.name for d in TRAIN_DIR.iterdir() if d.is_dir()])
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
print(f"
Found {NUM_CLASSES} classes: {CLASS_NAMES[:5]} ...")

In [ ]:
def count_images(split_dir: pathlib.Path) -> dict:
    """Return {class_name: count} for a split directory."""
    counts = {}
    for cls_dir in sorted(split_dir.iterdir()):
        if cls_dir.is_dir():
            counts[cls_dir.name] = sum(
                1 for f in cls_dir.iterdir()
                if f.suffix.lower() in _IMAGE_EXTS
            )
    return counts

train_counts = count_images(TRAIN_DIR)
val_counts   = count_images(VAL_DIR) if VAL_DIR else {}
test_counts  = count_images(TEST_DIR) if TEST_DIR else {}

print(f"Train images : {sum(train_counts.values())}")
print(f"Val images   : {sum(val_counts.values())}")
print(f"Test images  : {sum(test_counts.values())}")

In [ ]:
# Build full training file list
_all_files, _all_labels = [], []
for cls_name in CLASS_NAMES:
    cls_dir = TRAIN_DIR / cls_name
    for f in cls_dir.iterdir():
        if f.suffix.lower() in _IMAGE_EXTS:
            _all_files.append(str(f))
            _all_labels.append(CLASS_TO_IDX[cls_name])

# Carve test set from training data if no test dir exists
if TEST_DIR is None:
    print("No test/ directory found — carving 15% of training data as test set (stratified).")
    train_files, test_files, train_lbls, test_lbls = train_test_split(
        _all_files, _all_labels, test_size=0.15, random_state=SEED, stratify=_all_labels
    )
else:
    train_files, train_lbls = _all_files, _all_labels
    test_files, test_lbls = [], []
    for cls_name in CLASS_NAMES:
        test_cls = TEST_DIR / cls_name
        if test_cls.is_dir():
            for f in test_cls.iterdir():
                if f.suffix.lower() in _IMAGE_EXTS:
                    test_files.append(str(f))
                    test_lbls.append(CLASS_TO_IDX[cls_name])

# Carve val set from training data if no val dir exists
if VAL_DIR is None:
    print("No val/ directory found — carving 15% of training data as validation set (stratified).")
    train_files, val_files, train_lbls, val_lbls = train_test_split(
        train_files, train_lbls, test_size=0.15, random_state=SEED, stratify=train_lbls
    )
else:
    val_files, val_lbls = [], []
    for cls_name in CLASS_NAMES:
        val_cls = VAL_DIR / cls_name
        if val_cls.is_dir():
            for f in val_cls.iterdir():
                if f.suffix.lower() in _IMAGE_EXTS:
                    val_files.append(str(f))
                    val_lbls.append(CLASS_TO_IDX[cls_name])

print(f"
Final split sizes:")
print(f"  Train : {len(train_files)}")
print(f"  Val   : {len(val_files)}")
print(f"  Test  : {len(test_files)}")

## Section 3 — Exploratory Data Analysis (EDA)
### 3.1 Class Distribution

In [ ]:
import pandas as pd

train_series = pd.Series(train_counts).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 18))
ax.barh(train_series.index, train_series.values, color='steelblue')
ax.set_xlabel("Number of Images")
ax.set_title("Training Images per Class (sorted)")
ax.set_ylabel("Class")
plt.tight_layout()
plt.savefig("figures/class_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

stats = train_series.describe()
print(f"Min: {int(stats['min'])} | Max: {int(stats['max'])} | "
      f"Mean: {stats['mean']:.1f} | Median: {stats['50%']:.1f}")
imbalance_ratio = int(stats['max']) / max(int(stats['min']), 1)
print(f"Imbalance ratio (max/min): {imbalance_ratio:.1f}x")
if imbalance_ratio > 3:
    print("⚠️  Class imbalance detected — consider class-weighted loss or oversampling.")

### 3.2 Sample Image Grid

In [ ]:
set_seed()
_sample_indices = random.sample(range(len(train_files)), min(25, len(train_files)))
_sample_files  = [train_files[i] for i in _sample_indices]
_sample_labels = [CLASS_NAMES[train_lbls[i]] for i in _sample_indices]

fig, axes = plt.subplots(5, 5, figsize=(15, 15))
for ax, fpath, label in zip(axes.flat, _sample_files, _sample_labels):
    try:
        img = Image.open(fpath).convert("RGB")
        ax.imshow(img)
        ax.set_title(f"{label}
{img.size[0]}×{img.size[1]}", fontsize=7)
    except Exception:
        ax.set_title(f"{label}
(error)", fontsize=7, color='red')
    ax.axis('off')
fig.suptitle("Sample Training Images (5×5 Grid)", fontsize=14, y=1.01)
fig.tight_layout()
plt.savefig("figures/sample_grid.png", dpi=150, bbox_inches='tight')
plt.show()

### 3.3 Image Dimension Analysis

In [ ]:
set_seed()
_sample_500 = random.sample(range(len(train_files)), min(500, len(train_files)))
widths, heights = [], []
for i in tqdm(_sample_500, desc="Reading image dimensions"):
    try:
        w, h = Image.open(train_files[i]).size
        widths.append(w)
        heights.append(h)
    except Exception:
        pass

ratios = [w / h for w, h in zip(widths, heights)]
aspect_buckets = [
    "square" if 0.9 <= r <= 1.1 else "portrait" if r < 0.9 else "landscape"
    for r in ratios
]
colour_map = {"square": "steelblue", "portrait": "tomato", "landscape": "seagreen"}
colours = [colour_map[b] for b in aspect_buckets]

fig, ax = plt.subplots(figsize=(8, 6))
for bucket, col in colour_map.items():
    mask = [b == bucket for b in aspect_buckets]
    ax.scatter(
        [w for w, m in zip(widths, mask) if m],
        [h for h, m in zip(heights, mask) if m],
        c=col, label=bucket, alpha=0.5, s=20
    )
ax.set_xlabel("Width (px)")
ax.set_ylabel("Height (px)")
ax.set_title("Image Dimensions (sample of 500)")
ax.legend()
plt.tight_layout()
plt.savefig("figures/dimension_scatter.png", dpi=150, bbox_inches='tight')
plt.show()

from collections import Counter
res_counts = Counter(zip(widths, heights))
most_common_res, most_common_count = res_counts.most_common(1)[0]
print(f"Most common resolution: {most_common_res[0]}×{most_common_res[1]} "
      f"({most_common_count / len(widths) * 100:.1f}% of sample)")

### 3.4 Pixel Intensity Distribution

In [ ]:
set_seed()
_sample_200 = random.sample(range(len(train_files)), min(200, len(train_files)))

pixel_data = []
for i in tqdm(_sample_200, desc="Loading pixels"):
    try:
        img = np.array(Image.open(train_files[i]).convert("RGB").resize(IMG_SIZE)) / 255.0
        pixel_data.append(img.reshape(-1, 3))
    except Exception:
        pass

pixel_data = np.concatenate(pixel_data, axis=0)  # (N*H*W, 3)
CHANNEL_MEAN = pixel_data.mean(axis=0)
CHANNEL_STD  = pixel_data.std(axis=0)

print(f"Dataset mean (R,G,B): {CHANNEL_MEAN.round(4)}")
print(f"Dataset std  (R,G,B): {CHANNEL_STD.round(4)}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (name, col, ax) in enumerate(zip(["Red", "Green", "Blue"], ["red", "green", "blue"], axes)):
    ax.hist(pixel_data[:, i], bins=50, color=col, alpha=0.7, density=True)
    ax.set_title(f"{name} Channel")
    ax.set_xlabel("Pixel Value (normalised)")
    ax.set_ylabel("Density")
fig.suptitle("Per-Channel Pixel Intensity Distributions (200 training images)")
plt.tight_layout()
plt.savefig("figures/channel_histograms.png", dpi=150, bbox_inches='tight')
plt.show()

### 3.5 Class Similarity Heatmap

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def _class_colour_hist(cls_name: str, n: int = 20) -> np.ndarray:
    """Mean 64-bin colour histogram over the first n images of a class."""
    cls_files = [f for f, l in zip(train_files, train_lbls) if CLASS_NAMES[l] == cls_name][:n]
    hists = []
    for fp in cls_files:
        try:
            img = np.array(Image.open(fp).convert("RGB").resize((64, 64)))
            h = np.concatenate([
                np.histogram(img[:, :, c], bins=64, range=(0, 256))[0]
                for c in range(3)
            ]).astype(np.float32)
            hists.append(h / (h.sum() + 1e-8))
        except Exception:
            pass
    return np.mean(hists, axis=0) if hists else np.zeros(192, dtype=np.float32)

print("Computing class colour histograms (may take ~30s)...")
class_hists = np.stack([_class_colour_hist(c) for c in tqdm(CLASS_NAMES)])
sim_matrix = cosine_similarity(class_hists)

fig, ax = plt.subplots(figsize=(20, 18))
sns.heatmap(sim_matrix, xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            cmap="YlOrRd", ax=ax, vmin=0, vmax=1)
ax.set_title("Pairwise Class Colour-Histogram Cosine Similarity")
plt.xticks(fontsize=5, rotation=90)
plt.yticks(fontsize=5)
plt.tight_layout()
plt.savefig("figures/class_similarity_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()

## Section 4 — Data Preprocessing & Augmentation
### 4.1 Normalisation Values

In [ ]:
# Use dataset-computed mean/std; fall back to ImageNet if std is near zero (unreliable)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

if CHANNEL_STD.min() < 0.05:
    print("Dataset std values seem unreliable — falling back to ImageNet normalisation.")
    NORM_MEAN = IMAGENET_MEAN
    NORM_STD  = IMAGENET_STD
else:
    NORM_MEAN = CHANNEL_MEAN.tolist()
    NORM_STD  = CHANNEL_STD.tolist()
    print("Using dataset-computed normalisation.")

print(f"NORM_MEAN = {[f'{v:.4f}' for v in NORM_MEAN]}")
print(f"NORM_STD  = {[f'{v:.4f}' for v in NORM_STD]}")

### 4.2 – 4.3 Augmentation Pipelines

In [ ]:
train_transform = T.Compose([
    T.Resize(IMG_SIZE),
    T.RandomHorizontalFlip(),
    T.RandomRotation(15),
    T.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

eval_transform = T.Compose([
    T.Resize(IMG_SIZE),
    T.CenterCrop(IMG_SIZE),
    T.ToTensor(),
    T.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

print("Transforms defined: train (with augmentation), eval (normalise only).")

### 4.4 Custom Dataset & DataLoaders

In [ ]:
class FoodDataset(Dataset):
    def __init__(self, file_list: list, label_list: list, transform=None):
        self.files     = file_list
        self.labels    = label_list
        self.transform = transform

    def __len__(self) -> int:
        return len(self.files)

    def __getitem__(self, idx: int):
        img = Image.open(self.files[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

In [ ]:
train_ds = FoodDataset(train_files, train_lbls, transform=train_transform)
val_ds   = FoodDataset(val_files,   val_lbls,   transform=eval_transform)
test_ds  = FoodDataset(test_files,  test_lbls,  transform=eval_transform)

_loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

train_loader = DataLoader(train_ds, shuffle=True,  **_loader_kwargs)
val_loader   = DataLoader(val_ds,   shuffle=False, **_loader_kwargs)
test_loader  = DataLoader(test_ds,  shuffle=False, **_loader_kwargs)

print(f"DataLoaders ready — train: {len(train_loader)} batches, "
      f"val: {len(val_loader)} batches, test: {len(test_loader)} batches")